In [1]:
import time
import numpy as np
import pandas as pd
from constants import *
from run import *
from models import *
from sim import *
import pandas as pd
from sklearn.metrics import mean_squared_error

/Users/juar705/miniconda3/envs/bacterai/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
"""import AA file for use."""
csv_file_path = "/Users/juar705/Downloads/mock_data.csv"
df = pd.read_csv(csv_file_path)
truncated_df = df.head(1500)
del truncated_df['environment']

In [3]:
"""seperate into X_train and y_train sets
    X_train will be the amino acid columns 
    y_train will be the growth column"""

AA_columns = AA_SHORT
growth_columns = 'growth'

X_train = truncated_df[AA_columns].to_numpy()
y_train = truncated_df[growth_columns].to_numpy()

In [4]:
"""Train the models on the data from all previous rounds (excluding Round 1)"""

with open('config.json', 'r') as file:
    config = json.load(file)
    
n_ingredients = len(AA_SHORT)
MODEL_TYPE = ModelType(config["model_type"])   
TRANSFER_MODEL_FOLDER = config.get("transfer_model_folder", None)


MODEL_TYPE == ModelType.NEURAL_NET
transfer_model = NeuralNetModel.load_trained_models(TRANSFER_MODEL_FOLDER)
transfer_models = transfer_model.models if TRANSFER_MODEL_FOLDER else []
N_BAGS = config.get("n_bags", 25)

EXPT_FOLDER = config["experiment_path"]
MODEL_TYPE = ModelType(config["model_type"])
new_round_folder = os.path.join(EXPT_FOLDER, f"Round 1")
models_folder = os.path.join(new_round_folder, f"nn_models")
model = NeuralNetModel(models_folder)

model.train(
            X_train,
            y_train,
            n_ingredients=n_ingredients,
            n_bags=N_BAGS,
            bag_proportion=1.0,
            epochs=50,
            batch_size=20,
            lr=0.001,
            transfer_models=transfer_models,
        )


Bag 0, p=1.00
	EPOCH  1/50 | Train Loss: 0.1438, Train MSE: 0.1438
	EPOCH  2/50 | Train Loss: 0.0777, Train MSE: 0.0777
	EPOCH  3/50 | Train Loss: 0.0611, Train MSE: 0.0611
	EPOCH  4/50 | Train Loss: 0.0517, Train MSE: 0.0517
	EPOCH  5/50 | Train Loss: 0.0415, Train MSE: 0.0415
	EPOCH  6/50 | Train Loss: 0.0382, Train MSE: 0.0382
	EPOCH  7/50 | Train Loss: 0.0340, Train MSE: 0.0340
	EPOCH  8/50 | Train Loss: 0.0285, Train MSE: 0.0285
	EPOCH  9/50 | Train Loss: 0.0249, Train MSE: 0.0249
	EPOCH 10/50 | Train Loss: 0.0210, Train MSE: 0.0210
	EPOCH 11/50 | Train Loss: 0.0183, Train MSE: 0.0183
	EPOCH 12/50 | Train Loss: 0.0160, Train MSE: 0.0160
	EPOCH 13/50 | Train Loss: 0.0146, Train MSE: 0.0146
	EPOCH 14/50 | Train Loss: 0.0127, Train MSE: 0.0127
	EPOCH 15/50 | Train Loss: 0.0105, Train MSE: 0.0105
	EPOCH 16/50 | Train Loss: 0.0091, Train MSE: 0.0091
	EPOCH 17/50 | Train Loss: 0.0079, Train MSE: 0.0079
	EPOCH 18/50 | Train Loss: 0.0061, Train MSE: 0.0061
	EPOCH 19/50 | Train Loss: 0.00

In [ ]:
""" Create an array of ones for down direction or 0 for up direction"""

def media_array(n_ingredients, direction):
    if direction == SimDirection.DOWN:
        media = np.ones(n_ingredients)
    elif direction == SimDirection.UP:
        media = np.zeros(n_ingredients)
    else:
        raise ValueError("Error") 
    return media

batch_size = config["batch_size"]
DIRECTION = SimDirection(0)  

media = media_array(n_ingredients, DIRECTION)

Media Array:
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [6]:
def make_batch(
    model,
    media,
    new_round_n,
    batch_size,
    sim_types,
    rollout_trajectories,
    threshold,
    timeout=60,
    unique=True,
    direction=SimDirection.DOWN,
    go_beyond_frontier=True,
    used_experiments=None,
    redo_experiments=None,
):
    """ Make a new BacterAI batch; the main function that calls the simulation loops. """
    sim_types=sim_types
    n_types = len(sim_types)
    n_exps = batch_size // n_types
    batch_set = used_experiments
    sub_batches = []
    all_metrics = {}
    for idx, sim_type in enumerate(sim_types):
        if idx == n_types - 1:
            n_exps = batch_size - sum([len(x) for x in sub_batches])
        print(idx, sim_type, batch_size, n_exps, sum([len(x) for x in sub_batches]))
        batch, batch_set, metrics = perform_simulations(
            model,
            media.copy(),
            n_exps,
            threshold,
            sim_type,
            direction,
            new_round_n,
            unique=unique,
            timeout=timeout,
            batch_set=batch_set,
            n_rollout_trajectories=rollout_trajectories,
            go_beyond_frontier=go_beyond_frontier,
        )
        sub_batches.append(batch)
        all_metrics[sim_type.name] = metrics

    batch = pd.concat([redo_experiments] + sub_batches, ignore_index=True)
    return batch, batch_set, all_metrics

In [7]:
class NeuralNetModel(Model):
    def __init__(self, models_path):
        self.models_path = models_path
        self.models = []
        self.is_trained = False
        super().__init__(self, ModelType.NEURAL_NET)

    @classmethod
    def load_trained_models(cls, models_path):
        obj = cls(models_path)

        for filename in os.listdir(models_path):
            if "bag_model" in filename:
                model = torch.load(os.path.join(models_path, filename), map_location=torch.device(net.DEVICE), weights_only=False)
                obj.models.append(model)

        obj.is_trained = True
        return obj

    def check_path(self):
        if not os.path.exists(self.models_path):
            os.makedirs(self.models_path)

    def train(self, X_train, y_train, **kwargs):
        self.check_path()
        self.models = net.train_bagged(X_train, y_train, self.models_path, **kwargs)
        self.is_trained = True

    def evaluate(self, X, clip=True):
        if not self.is_trained:
            raise Exception("Neural net model needs to be trained before evaluating.")

        predictions, variances = net.eval_bagged(X, self.models)
        if clip:
            predictions = np.clip(predictions, 0, 1)
        return predictions, variances

In [ ]:
""" Trained set for batch if using pre-trained data
    If no pre trained data, use 'None' for used_experiments and redo_experiments"""

trained_set = pd.DataFrame(np.hstack((X_train, y_train.reshape(-1,1))))
used_experiments = set(map(tuple,trained_set.to_numpy()))
batch_data= used_experiments
#set parameters needed for make batch function
model = NeuralNetModel.load_trained_models(models_folder)
sim_types=[SimType(2)]
rollout_trajectories=config["n_rollouts"]
threshold=config['grow_threshold']
timeout=60 * 40
unique=True 
direction = SimDirection(0)
go_beyond_frontier=config['beyond_frontier']

# Make batch the main function that calls to make all simulations from perform simulations function
batch, batch_set, all_metrics = make_batch(
    model=model,
    media=media,
    new_round_n=2,
    batch_size=50,
    sim_types=sim_types,
    rollout_trajectories=rollout_trajectories,
    threshold=threshold,
    timeout=timeout,
    unique=unique,
    direction=direction,
    go_beyond_frontier=go_beyond_frontier,
    used_experiments=None,
    redo_experiments=None,)

0 SimType.ROLLOUT 50 50 0


Performing ROLLOUT Sims (DOWN) (1 loops):   2%|▏         | 1/50 [00:00<00:37,  1.32it/s]


	ADDED: [0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - FRONTIER

	ADDED: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - BEYOND

	EXISTS: [0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - FRONTIER

	EXISTS: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - BEYOND

	EXISTS: [0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - FRONTIER

	EXISTS: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - BEYOND

	EXISTS: [0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - FRONTIER

	EXISTS: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - BEYOND


Performing ROLLOUT Sims (DOWN) (5 loops):   6%|▌         | 3/50 [00:03<00:59,  1.27s/it]


	ADDED: [0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0] - FRONTIER

	ADDED: [0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - BEYOND

	EXISTS: [0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - FRONTIER

	EXISTS: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - BEYOND

	EXISTS: [0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - FRONTIER

	EXISTS: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - BEYOND

	EXISTS: [0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - FRONTIER

	EXISTS: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - BEYOND

	EXISTS: [0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - FRONTIER

	EXISTS: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - BEYOND


...


	ADDED: [0 0 0 0 0 1 0 0 0 1 0 0 0 0 1 0 0 0 0 0] - FRONTIER

	EXISTS: [0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0] - BEYOND

	EXISTS: [0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0] - FRONTIER

	EXISTS: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - BEYOND

	EXISTS: [0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - FRONTIER

	EXISTS: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - BEYOND

	EXISTS: [0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0] - FRONTIER

	EXISTS: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - BEYOND

	EXISTS: [0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - FRONTIER

	EXISTS: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - BEYOND

	EXISTS: [0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0] - FRONTIER

	EXISTS: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - BEYOND

	EXISTS: [0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - FRONTIER

	EXISTS: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - BEYOND

	EXISTS: [0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - FRONTIER

	EXISTS: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - BEYOND

	EXISTS:

Performing ROLLOUT Sims (DOWN) (3150 loops):  88%|████████▊ | 44/50 [36:03<14:32, 145.47s/it]


	ADDED: [0 0 0 1 1 1 0 0 1 0 1 0 0 1 0 0 0 1 0 0] - FRONTIER

	ADDED: [0 0 0 1 1 1 0 0 1 0 0 0 0 1 0 0 0 1 0 0] - BEYOND

	EXISTS: [0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - FRONTIER

	EXISTS: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - BEYOND

	EXISTS: [0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - FRONTIER

	EXISTS: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - BEYOND

	EXISTS: [0 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - FRONTIER

	EXISTS: [0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - BEYOND

	EXISTS: [0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - FRONTIER

	EXISTS: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - BEYOND

	EXISTS: [0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - FRONTIER

	EXISTS: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - BEYOND

	EXISTS: [0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - FRONTIER

	EXISTS: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - BEYOND

	EXISTS: [0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - FRONTIER

	EXISTS: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - BEYOND

	EXISTS: 

Performing ROLLOUT Sims (DOWN) (3497 loops):  90%|█████████ | 45/50 [40:00<04:26, 53.34s/it] 


	EXISTS: [0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - FRONTIER

	EXISTS: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0] - BEYOND
perform_simulations function took 2400132.42 ms


In [9]:
print(f"Batch:\n,{batch}")
display(batch)

Batch:
,    0  1  2  3  4  5  6  7  8  9  ...  17  18  19     type  direction  \
0   0  0  0  0  0  1  0  0  0  0  ...   0   0   0  ROLLOUT       DOWN   
1   0  0  0  0  0  0  0  0  0  0  ...   0   0   0  ROLLOUT       DOWN   
2   0  0  0  1  0  0  1  0  0  0  ...   0   0   0  ROLLOUT       DOWN   
3   0  0  0  1  0  0  0  0  0  0  ...   0   0   0  ROLLOUT       DOWN   
4   0  0  1  0  0  1  0  0  0  0  ...   0   0   0  ROLLOUT       DOWN   
5   0  0  1  0  0  0  0  0  0  0  ...   0   0   0  ROLLOUT       DOWN   
6   0  1  0  0  0  1  0  0  0  0  ...   0   0   0  ROLLOUT       DOWN   
7   0  1  0  0  0  0  0  0  0  0  ...   0   0   0  ROLLOUT       DOWN   
8   0  0  1  0  0  0  1  0  0  0  ...   0   0   0  ROLLOUT       DOWN   
9   0  0  0  0  1  0  1  0  0  0  ...   0   0   0  ROLLOUT       DOWN   
10  0  0  0  0  1  0  0  0  0  0  ...   0   0   0  ROLLOUT       DOWN   
11  0  0  0  0  0  0  1  0  0  0  ...   0   0   0  ROLLOUT       DOWN   
12  1  0  0  0  0  0  1  0  0  0  ...   0  

,0,1,2,3,4,5,6,7,8,9,...,17,18,19,type,direction,frontier_type,growth_pred,var,is_redo,round
0,0,0,0,0,0,1,0,0,0,0,...,0,0,0,ROLLOUT,DOWN,FRONTIER,0.350565,0.053519,False,2
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,ROLLOUT,DOWN,BEYOND,0.063990,0.002343,False,2
2,0,0,0,1,0,0,1,0,0,0,...,0,0,0,ROLLOUT,DOWN,FRONTIER,0.730090,0.050082,False,2
3,0,0,0,1,0,0,0,0,0,0,...,0,0,0,ROLLOUT,DOWN,BEYOND,0.055029,0.002685,False,2
4,0,0,1,0,0,1,0,0,0,0,...,0,0,0,ROLLOUT,DOWN,FRONTIER,0.404643,0.057461,False,2
5,0,0,1,0,0,0,0,0,0,0,...,0,0,0,ROLLOUT,DOWN,BEYOND,0.040942,0.002016,False,2
6,0,1,0,0,0,1,0,0,0,0,...,0,0,0,ROLLOUT,DOWN,FRONTIER,0.383070,0.043899,False,2
7,0,1,0,0,0,0,0,0,0,0,...,0,0,0,ROLLOUT,DOWN,BEYOND,0.026920,0.001932,False,2
8,0,0,1,0,0,0,1,0,0,0,...,0,0,0,ROLLOUT,DOWN,FRONTIER,0.539864,0.056562,False,2
9,0,0,0,0,1,0,1,0,0,0,...,0,0,0,ROLLOUT,DOWN,FRONTIER,0.616829,0.030724,False,2


In [10]:
print(f"Batch Set:\n{batch_set}")

Batch Set:
{(np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0)), (np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0)), (np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0)), (np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int

In [11]:
print(f"Metrics:\n{all_metrics}")

Metrics:
{'ROLLOUT': {'k_history': [], 'count_history': [], 'k_avg': 'n/a', 'count_avg': 'n/a', 'total_loops_count': 3497, 'time_to_finish_sec': 2400.13}}
